In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# 03_patient_aggregation_and_reference_sensitivity.ipynb
# COMPLETE SINGLE-CELL CODE
# ============================================================
#
# Purpose
# -------
# 1. Compare patient-level prediction aggregation rules using
#    development-only OOF predictions.
# 2. Select and freeze one aggregation rule using modal_low as
#    the primary derived patient-level reference.
# 3. Quantify uncertainty between the top two aggregation rules
#    using paired stratified patient bootstrap.
# 4. Evaluate sensitivity to alternative derived patient
#    reference definitions.
# 5. Generate a full aggregation-rule × reference-rule grid.
#
# Input predictions
# -----------------
# Two ViT loss candidates × three seeds:
#   - seed 42
#   - seed 2025
#   - seed 2026
#
# Total source prediction sets: 6
#
# Important
# ---------
# - The official LIMUC test set is never defined or loaded.
# - Aggregation selection uses modal_low only.
# - Alternative patient references are sensitivity analyses
#   only and cannot alter the locked aggregation decision.
# - No model retraining occurs in this notebook.
# ============================================================


# ============================================================
# 0. Install dependencies
# ============================================================


%pip -q install scikit-learn==1.6.1 pandas==2.2.2


# ============================================================
# 1. Imports
# ============================================================

import os
import json
import hashlib
import platform
import warnings

from itertools import combinations
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

from IPython.display import display


warnings.filterwarnings("ignore")


# ============================================================
# 2. Development-only safety check
# ============================================================

FORBIDDEN_TEST_OBJECTS = [
    "TEST_PATH",
    "TEST_ROOT",
    "test_df",
    "test_dataset",
    "test_loader",
    "test_predictions",
    "test_patient_predictions",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
    "ensemble_test_metrics",
    "ensemble_test_patient_metrics",
]

present_test_objects = [
    object_name
    for object_name in FORBIDDEN_TEST_OBJECTS
    if object_name in globals()
]

if present_test_objects:
    raise RuntimeError(
        "Official-test objects exist in the current runtime: "
        f"{present_test_objects}\n"
        "Restart the runtime and run only this cell."
    )

print("Development-only safety check passed.")
print("Official LIMUC test data are not defined or loaded.")


# ============================================================
# 3. Mount Google Drive
# ============================================================

from google.colab import drive

(drive.mount(
    "/content/drive",
    force_remount=True,
) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 4. Paths
# ============================================================

DEVELOPMENT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/train_and_validation_sets')
)

SCREENING_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/01_vit_loss_screening_development_only')
)

CONFIRMATION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/02_vit_loss_multiseed_confirmation_epoch_resume_v2')
)

OUTPUT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/03_patient_aggregation_and_reference_sensitivity')
)

MANIFEST_PATH = (
    SCREENING_ROOT
    / "development_image_manifest_with_folds.csv"
)

FOLD_ASSIGNMENT_PATH = (
    SCREENING_ROOT
    / "patient_fold_assignment.csv"
)

REFERENCE_AUDIT_PATH = (
    SCREENING_ROOT
    / "patient_reference_rule_sensitivity.csv"
)

SCREENING_PLAN_PATH = (
    SCREENING_ROOT
    / "analysis_plan_frozen_before_training.json"
)

CONFIRMATION_PLAN_PATH = (
    CONFIRMATION_ROOT
    / "analysis_plan_frozen_before_confirmation.json"
)

required_paths = [
    DEVELOPMENT_ROOT,
    SCREENING_ROOT,
    CONFIRMATION_ROOT,
    MANIFEST_PATH,
    FOLD_ASSIGNMENT_PATH,
    REFERENCE_AUDIT_PATH,
    SCREENING_PLAN_PATH,
    CONFIRMATION_PLAN_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist:\n{required_path}"
        )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nDevelopment root:", DEVELOPMENT_ROOT)
print("Screening root:", SCREENING_ROOT)
print("Confirmation root:", CONFIRMATION_ROOT)
print("Aggregation output:", OUTPUT_ROOT)


# ============================================================
# 5. Frozen analysis configuration
# ============================================================

PROTOCOL_VERSION = (
    "patient_aggregation_reference_sensitivity_v1"
)

PRIMARY_REFERENCE_RULE = "modal_low"

REFERENCE_RULES = {
    "modal_low": (
        "Modal image-level MES with lower-class "
        "tie-breaking; primary reference"
    ),

    "modal_high": (
        "Modal image-level MES with higher-class "
        "tie-breaking"
    ),

    "median_low": (
        "Floor of the median image-level MES"
    ),

    "median_high": (
        "Ceiling of the median image-level MES"
    ),

    "maximum_label": (
        "Maximum observed image-level MES"
    ),
}

AGGREGATION_RULES = {
    "mean_probability_argmax": {
        "display_name": (
            "Mean class probability, argmax"
        ),

        "description": (
            "Average image-level class probabilities within "
            "each patient and select the class with the largest "
            "mean probability."
        ),
    },

    "median_probability_argmax": {
        "display_name": (
            "Median class probability, argmax"
        ),

        "description": (
            "Calculate the median probability for each class "
            "within each patient, renormalize, and select the "
            "class with the largest median probability."
        ),
    },

    "majority_vote_low_tie": {
        "display_name": (
            "Majority vote, lower-class tie"
        ),

        "description": (
            "Select the most frequent image-level predicted "
            "class within each patient; select the lower class "
            "when multiple classes are tied."
        ),
    },

    "mean_expected_mes_round": {
        "display_name": (
            "Mean expected MES, rounded"
        ),

        "description": (
            "Calculate the expected MES for each image, average "
            "within each patient, and round half upward to the "
            "nearest class."
        ),
    },

    "median_expected_mes_round": {
        "display_name": (
            "Median expected MES, rounded"
        ),

        "description": (
            "Calculate the expected MES for each image, take "
            "the patient median, and round half upward to the "
            "nearest class."
        ),
    },

    "maximum_predicted_class": {
        "display_name": (
            "Maximum predicted image severity"
        ),

        "description": (
            "Assign each patient the maximum image-level "
            "predicted MES class."
        ),
    },
}

CANDIDATES = [
    "vit_L5_expected_ordinal_distance_a1_w020",
    "vit_L2_expected_value_ordinal_mae_w010",
]

SEEDS = [
    42,
    2025,
    2026,
]

NUM_CLASSES = 4

SELECTION_HIERARCHY = [
    "Mean patient QWK across all six OOF sources: higher",
    "Mean patient MAE across all six OOF sources: lower",
    "Mean patient macro F1 across all six OOF sources: higher",
    "Mean patient accuracy across all six OOF sources: higher",
]

BOOTSTRAP_REPETITIONS = 5000
BOOTSTRAP_SEED = 20260801


# ============================================================
# 6. File and serialization helpers
# ============================================================

def file_sha256(
    path: Path,
) -> str:
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            hasher.update(block)

    return hasher.hexdigest()


def dictionary_sha256(
    dictionary: dict,
) -> str:
    encoded = json.dumps(
        dictionary,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def temporary_path(
    path: Path,
) -> Path:
    return path.with_name(
        path.name + ".temporary"
    )


def atomic_json_save(
    payload: dict,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=2,
            default=str,
        )

    os.replace(
        temp_path,
        path,
    )


def atomic_csv_save(
    dataframe: pd.DataFrame,
    path: Path,
    index: bool = False,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    dataframe.to_csv(
        temp_path,
        index=index,
    )

    os.replace(
        temp_path,
        path,
    )


def safe_package_version(
    package_name: str,
):
    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


# Remove abandoned temporary files from a previous interruption.
for temporary_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        temporary_file.unlink()
    except Exception:
        pass


# ============================================================
# 7. Identify and hash all six OOF source files
# ============================================================

source_file_records = []

for candidate_name in CANDIDATES:
    for training_seed in SEEDS:
        source_id = (
            f"{candidate_name}"
            f"__seed_{training_seed}"
        )

        oof_path = (
            CONFIRMATION_ROOT
            / candidate_name
            / f"seed_{training_seed}"
            / "oof_image_predictions.csv"
        )

        seed_result_path = (
            CONFIRMATION_ROOT
            / candidate_name
            / f"seed_{training_seed}"
            / "seed_result.json"
        )

        if not oof_path.exists():
            raise FileNotFoundError(
                "Required OOF prediction file is missing:\n"
                f"{oof_path}"
            )

        if not seed_result_path.exists():
            raise FileNotFoundError(
                "Required seed result file is missing:\n"
                f"{seed_result_path}"
            )

        source_file_records.append({
            "source_id": source_id,
            "candidate_name": candidate_name,
            "training_seed": int(
                training_seed
            ),
            "oof_prediction_path": str(
                oof_path
            ),
            "oof_prediction_sha256": (
                file_sha256(oof_path)
            ),
            "seed_result_path": str(
                seed_result_path
            ),
            "seed_result_sha256": (
                file_sha256(
                    seed_result_path
                )
            ),
        })

source_file_df = pd.DataFrame(
    source_file_records
)

if len(source_file_df) != 6:
    raise RuntimeError(
        "Exactly six OOF source files were expected."
    )

print("\nVerified OOF source files:")

display(
    source_file_df[
        [
            "source_id",
            "candidate_name",
            "training_seed",
            "oof_prediction_path",
        ]
    ]
)


# ============================================================
# 8. Freeze aggregation analysis plan
# ============================================================

immutable_analysis_plan = {
    "protocol_version": (
        PROTOCOL_VERSION
    ),

    "study_stage": (
        "Development-only patient aggregation selection "
        "and patient-reference sensitivity analysis"
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,

    "primary_reference_rule": (
        PRIMARY_REFERENCE_RULE
    ),

    "reference_rules": (
        REFERENCE_RULES
    ),

    "aggregation_rules": (
        AGGREGATION_RULES
    ),

    "selection_hierarchy": (
        SELECTION_HIERARCHY
    ),

    "selection_sources": (
        source_file_records
    ),

    "selection_source_weighting": (
        "Each of the six candidate-seed OOF prediction "
        "sets receives equal weight."
    ),

    "aggregation_selection_scope": (
        "Aggregation is selected using modal_low only. "
        "Alternative patient references are sensitivity "
        "analyses and cannot modify the selected rule."
    ),

    "bootstrap_repetitions": int(
        BOOTSTRAP_REPETITIONS
    ),

    "bootstrap_seed": int(
        BOOTSTRAP_SEED
    ),

    "development_manifest_path": str(
        MANIFEST_PATH
    ),

    "development_manifest_sha256": (
        file_sha256(
            MANIFEST_PATH
        )
    ),

    "fold_assignment_path": str(
        FOLD_ASSIGNMENT_PATH
    ),

    "fold_assignment_sha256": (
        file_sha256(
            FOLD_ASSIGNMENT_PATH
        )
    ),

    "reference_audit_path": str(
        REFERENCE_AUDIT_PATH
    ),

    "reference_audit_sha256": (
        file_sha256(
            REFERENCE_AUDIT_PATH
        )
    ),

    "screening_plan_path": str(
        SCREENING_PLAN_PATH
    ),

    "screening_plan_sha256": (
        file_sha256(
            SCREENING_PLAN_PATH
        )
    ),

    "confirmation_plan_path": str(
        CONFIRMATION_PLAN_PATH
    ),

    "confirmation_plan_sha256": (
        file_sha256(
            CONFIRMATION_PLAN_PATH
        )
    ),

    "interpretive_constraint": (
        "Maximum predicted severity is not treated as "
        "clinically invalid solely because it performs poorly "
        "against a modal reference. Its compatibility with "
        "the maximum-label reference is evaluated separately."
    ),
}

analysis_plan_fingerprint = (
    dictionary_sha256(
        immutable_analysis_plan
    )
)

analysis_plan = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    **immutable_analysis_plan,
}

analysis_plan_path = (
    OUTPUT_ROOT
    / "analysis_plan_frozen_before_aggregation.json"
)

if analysis_plan_path.exists():
    with open(
        analysis_plan_path,
        "r",
        encoding="utf-8",
    ) as file:
        existing_plan = json.load(
            file
        )

    if (
        existing_plan.get(
            "analysis_plan_fingerprint"
        )
        !=
        analysis_plan_fingerprint
    ):
        raise RuntimeError(
            "The existing aggregation analysis plan differs "
            "from the current configuration.\n"
            "Do not mix outputs from different protocols. "
            "Use a new versioned OUTPUT_ROOT."
        )

    print(
        "\nExisting aggregation analysis plan validated:",
        analysis_plan_path,
    )

else:
    atomic_json_save(
        analysis_plan,
        analysis_plan_path,
    )

    print(
        "\nSaved frozen aggregation analysis plan:",
        analysis_plan_path,
    )

print(
    "Analysis-plan fingerprint:",
    analysis_plan_fingerprint,
)


# ============================================================
# 9. Environment metadata
# ============================================================

environment_metadata = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python": platform.python_version(),
    "platform": platform.platform(),

    "numpy": np.__version__,
    "pandas": pd.__version__,

    "scikit_learn": (
        safe_package_version(
            "scikit-learn"
        )
    ),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

atomic_json_save(
    environment_metadata,
    OUTPUT_ROOT
    / "environment_metadata.json",
)


# ============================================================
# 10. Load and validate frozen development metadata
# ============================================================

manifest_df = pd.read_csv(
    MANIFEST_PATH
)

required_manifest_columns = {
    "relative_path",
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
}

missing_manifest_columns = (
    required_manifest_columns
    -
    set(manifest_df.columns)
)

if missing_manifest_columns:
    raise RuntimeError(
        "Development manifest is missing columns: "
        f"{sorted(missing_manifest_columns)}"
    )

manifest_df["path"] = (
    manifest_df["relative_path"]
    .map(
        lambda relative_path: str(
            DEVELOPMENT_ROOT
            / Path(relative_path)
        )
    )
)

for integer_column in [
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
]:
    manifest_df[integer_column] = (
        manifest_df[integer_column]
        .astype(int)
    )

manifest_df = (
    manifest_df
    .sort_values("path")
    .reset_index(drop=True)
)

if len(manifest_df) != 9590:
    raise RuntimeError(
        f"Expected 9,590 development images; "
        f"found {len(manifest_df)}."
    )

if (
    manifest_df["patient_id"]
    .nunique()
    !=
    478
):
    raise RuntimeError(
        "Expected 478 development patients."
    )

if manifest_df["path"].duplicated().any():
    raise RuntimeError(
        "Duplicate paths exist in the development manifest."
    )

reference_audit_df = pd.read_csv(
    REFERENCE_AUDIT_PATH
)

required_reference_columns = {
    "patient_id",
    *REFERENCE_RULES.keys(),
    "n_images",
    "n_unique_labels",
    "label_range",
    "modal_tie",
    "majority_fraction",
    "label_entropy",
}

missing_reference_columns = (
    required_reference_columns
    -
    set(reference_audit_df.columns)
)

if missing_reference_columns:
    raise RuntimeError(
        "Patient-reference audit is missing columns: "
        f"{sorted(missing_reference_columns)}"
    )

reference_audit_df["patient_id"] = (
    reference_audit_df["patient_id"]
    .astype(int)
)

for reference_rule in REFERENCE_RULES:
    reference_audit_df[
        reference_rule
    ] = (
        reference_audit_df[
            reference_rule
        ]
        .astype(int)
    )

reference_audit_df = (
    reference_audit_df
    .sort_values("patient_id")
    .reset_index(drop=True)
)

if len(reference_audit_df) != 478:
    raise RuntimeError(
        "Patient-reference audit does not contain "
        "478 patients."
    )

if reference_audit_df[
    "patient_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate patient IDs exist in the reference audit."
    )

manifest_patient_df = (
    manifest_df[
        [
            "patient_id",
            "patient_y_true",
            "cv_fold",
        ]
    ]
    .drop_duplicates()
)

reference_validation_df = (
    manifest_patient_df
    .merge(
        reference_audit_df[
            [
                "patient_id",
                PRIMARY_REFERENCE_RULE,
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

if len(reference_validation_df) != 478:
    raise RuntimeError(
        "Manifest/reference validation did not align "
        "all patients."
    )

if not (
    reference_validation_df[
        "patient_y_true"
    ]
    ==
    reference_validation_df[
        PRIMARY_REFERENCE_RULE
    ]
).all():
    raise RuntimeError(
        "Manifest patient_y_true does not match modal_low."
    )

print("\nDevelopment images:", len(manifest_df))
print(
    "Development patients:",
    manifest_df["patient_id"].nunique(),
)
print(
    "Primary derived reference:",
    PRIMARY_REFERENCE_RULE,
)


# ============================================================
# 11. Metrics
# ============================================================

def compute_metrics(
    y_true,
    y_pred,
) -> dict:
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    if len(y_true) == 0:
        raise ValueError(
            "Cannot calculate metrics on an empty set."
        )

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred have different lengths."
        )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,
        labels=[
            0,
            1,
            2,
            3,
        ],
        weights="quadratic",
    )

    return {
        "qwk": (
            float(qwk)
            if np.isfinite(qwk)
            else np.nan
        ),

        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=[
                    0,
                    1,
                    2,
                    3,
                ],
                average="macro",
                zero_division=0,
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }


def make_confusion_df(
    y_true,
    y_pred,
) -> pd.DataFrame:
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1,
            2,
            3,
        ],
    )

    return pd.DataFrame(
        matrix,
        index=[
            "true_0",
            "true_1",
            "true_2",
            "true_3",
        ],
        columns=[
            "pred_0",
            "pred_1",
            "pred_2",
            "pred_3",
        ],
    )


# ============================================================
# 12. Validate one OOF source prediction file
# ============================================================

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


def validate_oof_source(
    prediction_df: pd.DataFrame,
    source_id: str,
    candidate_name: str,
    training_seed: int,
) -> pd.DataFrame:
    prediction_df = prediction_df.copy()

    required_columns = {
        "path",
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
        *PROBABILITY_COLUMNS,
    }

    missing_columns = (
        required_columns
        -
        set(prediction_df.columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"{source_id} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    prediction_df["path"] = (
        prediction_df["path"]
        .astype(str)
    )

    for integer_column in [
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
    ]:
        prediction_df[integer_column] = (
            prediction_df[integer_column]
            .astype(int)
        )

    if len(prediction_df) != 9590:
        raise RuntimeError(
            f"{source_id} contains {len(prediction_df)} rows; "
            "9,590 were expected."
        )

    if prediction_df["path"].duplicated().any():
        raise RuntimeError(
            f"{source_id} contains duplicate image paths."
        )

    probability_array = (
        prediction_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    if not np.isfinite(
        probability_array
    ).all():
        raise RuntimeError(
            f"{source_id} contains non-finite probabilities."
        )

    probability_sums = (
        probability_array.sum(
            axis=1
        )
    )

    if not np.allclose(
        probability_sums,
        1.0,
        atol=2e-3,
        rtol=0.0,
    ):
        raise RuntimeError(
            f"{source_id} has probability rows that "
            "do not sum to one."
        )

    prediction_df["y_pred"] = (
        probability_array
        .argmax(axis=1)
        .astype(int)
    )

    expected_metadata = (
        manifest_df[
            [
                "path",
                "patient_id",
                "label",
                "patient_y_true",
                "cv_fold",
            ]
        ]
        .rename(
            columns={
                "patient_id": (
                    "patient_id_expected"
                ),
                "label": (
                    "y_true_expected"
                ),
                "patient_y_true": (
                    "patient_y_true_expected"
                ),
                "cv_fold": (
                    "fold_expected"
                ),
            }
        )
    )

    validation_df = (
        prediction_df[
            [
                "path",
                "patient_id",
                "y_true",
                "patient_y_true",
            ]
        ]
        .merge(
            expected_metadata,
            on="path",
            how="inner",
            validate="one_to_one",
        )
    )

    if len(validation_df) != 9590:
        raise RuntimeError(
            f"{source_id} image paths do not match "
            "the frozen manifest."
        )

    if not (
        validation_df["patient_id"]
        ==
        validation_df[
            "patient_id_expected"
        ]
    ).all():
        raise RuntimeError(
            f"{source_id} patient IDs do not match "
            "the manifest."
        )

    if not (
        validation_df["y_true"]
        ==
        validation_df[
            "y_true_expected"
        ]
    ).all():
        raise RuntimeError(
            f"{source_id} image labels do not match "
            "the manifest."
        )

    if not (
        validation_df[
            "patient_y_true"
        ]
        ==
        validation_df[
            "patient_y_true_expected"
        ]
    ).all():
        raise RuntimeError(
            f"{source_id} patient references do not "
            "match the frozen manifest."
        )

    if "fold" not in prediction_df.columns:
        fold_map = (
            manifest_df[
                [
                    "path",
                    "cv_fold",
                ]
            ]
            .set_index("path")[
                "cv_fold"
            ]
            .to_dict()
        )

        prediction_df["fold"] = (
            prediction_df["path"]
            .map(fold_map)
            .astype(int)
        )

    else:
        prediction_df["fold"] = (
            prediction_df["fold"]
            .astype(int)
        )

    if (
        prediction_df
        .groupby("patient_id")[
            "fold"
        ]
        .nunique()
        .max()
        !=
        1
    ):
        raise RuntimeError(
            f"{source_id} places a patient in multiple "
            "OOF folds."
        )

    prediction_df["source_id"] = (
        source_id
    )

    prediction_df["candidate_name"] = (
        candidate_name
    )

    prediction_df["training_seed"] = int(
        training_seed
    )

    return (
        prediction_df
        .sort_values("path")
        .reset_index(drop=True)
    )


# ============================================================
# 13. Load all six validated OOF prediction sources
# ============================================================

source_predictions = {}

for source_record in source_file_records:
    source_id = source_record[
        "source_id"
    ]

    candidate_name = source_record[
        "candidate_name"
    ]

    training_seed = int(
        source_record[
            "training_seed"
        ]
    )

    oof_path = Path(
        source_record[
            "oof_prediction_path"
        ]
    )

    raw_prediction_df = pd.read_csv(
        oof_path
    )

    validated_prediction_df = (
        validate_oof_source(
            prediction_df=(
                raw_prediction_df
            ),

            source_id=source_id,

            candidate_name=(
                candidate_name
            ),

            training_seed=(
                training_seed
            ),
        )
    )

    source_predictions[
        source_id
    ] = validated_prediction_df

    print(
        "Validated:",
        source_id,
    )

if len(source_predictions) != 6:
    raise RuntimeError(
        "Six validated OOF prediction sources "
        "were expected."
    )


# ============================================================
# 14. Aggregation helpers
# ============================================================

CLASS_VALUES = np.arange(
    NUM_CLASSES,
    dtype=float,
)


def round_half_up_to_class(
    value,
) -> int:
    rounded = int(
        np.floor(
            float(value) + 0.5
        )
    )

    return int(
        np.clip(
            rounded,
            0,
            NUM_CLASSES - 1,
        )
    )


def lower_tie_majority_vote(
    values,
) -> int:
    values = np.asarray(
        values,
        dtype=int,
    )

    counts = np.bincount(
        values,
        minlength=NUM_CLASSES,
    )

    maximum_count = counts.max()

    tied_classes = np.flatnonzero(
        counts == maximum_count
    )

    return int(
        tied_classes.min()
    )


def aggregate_one_source(
    image_prediction_df: pd.DataFrame,
) -> pd.DataFrame:
    working_df = (
        image_prediction_df
        .copy()
    )

    probability_array = (
        working_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    working_df[
        "image_expected_mes"
    ] = (
        probability_array
        @ CLASS_VALUES
    )

    working_df[
        "image_hard_prediction"
    ] = (
        probability_array
        .argmax(axis=1)
        .astype(int)
    )

    patient_ids = sorted(
        working_df[
            "patient_id"
        ]
        .unique()
        .tolist()
    )

    output_rows = []

    reference_lookup = (
        reference_audit_df
        .set_index("patient_id")
    )

    for patient_id in patient_ids:
        patient_images = (
            working_df[
                working_df["patient_id"]
                ==
                patient_id
            ]
        )

        probabilities = (
            patient_images[
                PROBABILITY_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )

        hard_predictions = (
            patient_images[
                "image_hard_prediction"
            ]
            .to_numpy(
                dtype=int
            )
        )

        expected_values = (
            patient_images[
                "image_expected_mes"
            ]
            .to_numpy(
                dtype=float
            )
        )

        mean_probabilities = (
            probabilities.mean(
                axis=0
            )
        )

        median_probabilities = (
            np.median(
                probabilities,
                axis=0,
            )
        )

        median_probability_sum = (
            median_probabilities.sum()
        )

        if median_probability_sum > 0:
            normalized_median_probabilities = (
                median_probabilities
                /
                median_probability_sum
            )

        else:
            normalized_median_probabilities = (
                np.full(
                    NUM_CLASSES,
                    1.0 / NUM_CLASSES,
                )
            )

        aggregation_outputs = {
            "mean_probability_argmax": {
                "y_pred": int(
                    mean_probabilities.argmax()
                ),

                "continuous_score": float(
                    mean_probabilities
                    @ CLASS_VALUES
                ),
            },

            "median_probability_argmax": {
                "y_pred": int(
                    normalized_median_probabilities
                    .argmax()
                ),

                "continuous_score": float(
                    normalized_median_probabilities
                    @ CLASS_VALUES
                ),
            },

            "majority_vote_low_tie": {
                "y_pred": (
                    lower_tie_majority_vote(
                        hard_predictions
                    )
                ),

                "continuous_score": float(
                    hard_predictions.mean()
                ),
            },

            "mean_expected_mes_round": {
                "y_pred": (
                    round_half_up_to_class(
                        expected_values.mean()
                    )
                ),

                "continuous_score": float(
                    expected_values.mean()
                ),
            },

            "median_expected_mes_round": {
                "y_pred": (
                    round_half_up_to_class(
                        np.median(
                            expected_values
                        )
                    )
                ),

                "continuous_score": float(
                    np.median(
                        expected_values
                    )
                ),
            },

            "maximum_predicted_class": {
                "y_pred": int(
                    hard_predictions.max()
                ),

                "continuous_score": float(
                    hard_predictions.max()
                ),
            },
        }

        patient_reference_row = (
            reference_lookup.loc[
                int(patient_id)
            ]
        )

        for aggregation_rule, output in (
            aggregation_outputs.items()
        ):
            row = {
                "patient_id": int(
                    patient_id
                ),

                "aggregation_rule": (
                    aggregation_rule
                ),

                "aggregation_display_name": (
                    AGGREGATION_RULES[
                        aggregation_rule
                    ]["display_name"]
                ),

                "y_pred": int(
                    output["y_pred"]
                ),

                "continuous_score": float(
                    output[
                        "continuous_score"
                    ]
                ),

                "n_images": int(
                    len(patient_images)
                ),

                "mean_hard_prediction": float(
                    hard_predictions.mean()
                ),

                "maximum_hard_prediction": int(
                    hard_predictions.max()
                ),
            }

            for reference_rule in (
                REFERENCE_RULES
            ):
                row[reference_rule] = int(
                    patient_reference_row[
                        reference_rule
                    ]
                )

            output_rows.append(
                row
            )

    output_df = pd.DataFrame(
        output_rows
    )

    expected_rows = (
        478
        *
        len(
            AGGREGATION_RULES
        )
    )

    if len(output_df) != expected_rows:
        raise RuntimeError(
            "Unexpected number of patient aggregation rows.\n"
            f"Expected: {expected_rows}\n"
            f"Observed: {len(output_df)}"
        )

    return (
        output_df
        .sort_values(
            [
                "aggregation_rule",
                "patient_id",
            ]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 15. Aggregate every source using every candidate rule
# ============================================================

patient_prediction_parts = []

for source_record in source_file_records:
    source_id = source_record[
        "source_id"
    ]

    candidate_name = source_record[
        "candidate_name"
    ]

    training_seed = int(
        source_record[
            "training_seed"
        ]
    )

    aggregated_df = (
        aggregate_one_source(
            source_predictions[
                source_id
            ]
        )
    )

    aggregated_df["source_id"] = (
        source_id
    )

    aggregated_df["candidate_name"] = (
        candidate_name
    )

    aggregated_df["training_seed"] = (
        training_seed
    )

    patient_prediction_parts.append(
        aggregated_df
    )

patient_predictions_long_df = pd.concat(
    patient_prediction_parts,
    ignore_index=True,
)

expected_long_rows = (
    6
    *
    478
    *
    len(
        AGGREGATION_RULES
    )
)

if (
    len(patient_predictions_long_df)
    !=
    expected_long_rows
):
    raise RuntimeError(
        "Unexpected number of source-rule-patient rows.\n"
        f"Expected: {expected_long_rows}\n"
        f"Observed: {len(patient_predictions_long_df)}"
    )

patient_predictions_long_path = (
    OUTPUT_ROOT
    / "all_patient_predictions_by_source_and_aggregation.csv"
)

atomic_csv_save(
    patient_predictions_long_df,
    patient_predictions_long_path,
    index=False,
)


# ============================================================
# 16. Primary aggregation analysis using modal_low
# ============================================================

primary_result_rows = []

for (
    source_id,
    candidate_name,
    training_seed,
    aggregation_rule,
), group_df in (
    patient_predictions_long_df
    .groupby(
        [
            "source_id",
            "candidate_name",
            "training_seed",
            "aggregation_rule",
        ],
        sort=True,
    )
):
    metrics = compute_metrics(
        group_df[
            PRIMARY_REFERENCE_RULE
        ],
        group_df["y_pred"],
    )

    prediction_counts = (
        group_df["y_pred"]
        .value_counts()
        .reindex(
            range(NUM_CLASSES),
            fill_value=0,
        )
        .sort_index()
    )

    primary_result_rows.append({
        "source_id": source_id,
        "candidate_name": candidate_name,
        "training_seed": int(
            training_seed
        ),

        "aggregation_rule": (
            aggregation_rule
        ),

        "aggregation_display_name": (
            AGGREGATION_RULES[
                aggregation_rule
            ]["display_name"]
        ),

        "reference_rule": (
            PRIMARY_REFERENCE_RULE
        ),

        "qwk": float(
            metrics["qwk"]
        ),

        "mae": float(
            metrics["mae"]
        ),

        "macro_f1": float(
            metrics["macro_f1"]
        ),

        "accuracy": float(
            metrics["accuracy"]
        ),

        "n_predicted_0": int(
            prediction_counts.loc[0]
        ),

        "n_predicted_1": int(
            prediction_counts.loc[1]
        ),

        "n_predicted_2": int(
            prediction_counts.loc[2]
        ),

        "n_predicted_3": int(
            prediction_counts.loc[3]
        ),
    })

aggregation_results_by_source_df = (
    pd.DataFrame(
        primary_result_rows
    )
)

aggregation_results_by_source_path = (
    OUTPUT_ROOT
    / "aggregation_results_by_source.csv"
)

atomic_csv_save(
    aggregation_results_by_source_df,
    aggregation_results_by_source_path,
    index=False,
)


# ============================================================
# 17. Rank rules separately within every OOF source
# ============================================================

within_source_rank_parts = []

for source_id, source_df in (
    aggregation_results_by_source_df
    .groupby("source_id")
):
    ranked_source_df = (
        source_df
        .sort_values(
            by=[
                "qwk",
                "mae",
                "macro_f1",
                "accuracy",
            ],
            ascending=[
                False,
                True,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    ranked_source_df[
        "within_source_rank"
    ] = np.arange(
        1,
        len(ranked_source_df) + 1,
    )

    within_source_rank_parts.append(
        ranked_source_df
    )

within_source_rank_df = pd.concat(
    within_source_rank_parts,
    ignore_index=True,
)

within_source_rank_path = (
    OUTPUT_ROOT
    / "aggregation_within_source_ranks.csv"
)

atomic_csv_save(
    within_source_rank_df,
    within_source_rank_path,
    index=False,
)


# ============================================================
# 18. Summary across all six OOF sources
# ============================================================

aggregation_summary_df = (
    aggregation_results_by_source_df
    .groupby(
        [
            "aggregation_rule",
            "aggregation_display_name",
        ],
        as_index=False,
    )
    .agg(
        n_sources=(
            "source_id",
            "nunique",
        ),

        qwk_mean=(
            "qwk",
            "mean",
        ),

        qwk_std=(
            "qwk",
            "std",
        ),

        qwk_min=(
            "qwk",
            "min",
        ),

        qwk_max=(
            "qwk",
            "max",
        ),

        mae_mean=(
            "mae",
            "mean",
        ),

        mae_std=(
            "mae",
            "std",
        ),

        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),

        macro_f1_std=(
            "macro_f1",
            "std",
        ),

        accuracy_mean=(
            "accuracy",
            "mean",
        ),

        accuracy_std=(
            "accuracy",
            "std",
        ),
    )
)

rank_counts_df = (
    within_source_rank_df
    .groupby(
        "aggregation_rule",
        as_index=False,
    )
    .agg(
        n_source_wins=(
            "within_source_rank",
            lambda values: int(
                np.sum(
                    np.asarray(values)
                    ==
                    1
                )
            ),
        ),

        median_within_source_rank=(
            "within_source_rank",
            "median",
        ),

        mean_within_source_rank=(
            "within_source_rank",
            "mean",
        ),
    )
)

aggregation_summary_df = (
    aggregation_summary_df
    .merge(
        rank_counts_df,
        on="aggregation_rule",
        how="left",
        validate="one_to_one",
    )
)

numeric_columns = (
    aggregation_summary_df
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)

aggregation_summary_df[
    numeric_columns
] = (
    aggregation_summary_df[
        numeric_columns
    ]
    .fillna(0.0)
)

aggregation_ranking_df = (
    aggregation_summary_df
    .sort_values(
        by=[
            "qwk_mean",
            "mae_mean",
            "macro_f1_mean",
            "accuracy_mean",
        ],
        ascending=[
            False,
            True,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

aggregation_ranking_df.insert(
    0,
    "development_rank",
    np.arange(
        1,
        len(
            aggregation_ranking_df
        ) + 1,
    ),
)

aggregation_summary_path = (
    OUTPUT_ROOT
    / "aggregation_summary_across_six_sources.csv"
)

aggregation_ranking_path = (
    OUTPUT_ROOT
    / "aggregation_ranking_primary_modal_low.csv"
)

atomic_csv_save(
    aggregation_summary_df,
    aggregation_summary_path,
    index=False,
)

atomic_csv_save(
    aggregation_ranking_df,
    aggregation_ranking_path,
    index=False,
)


# ============================================================
# 19. Candidate-specific aggregation summary
# ============================================================

aggregation_summary_by_candidate_df = (
    aggregation_results_by_source_df
    .groupby(
        [
            "candidate_name",
            "aggregation_rule",
            "aggregation_display_name",
        ],
        as_index=False,
    )
    .agg(
        n_seeds=(
            "training_seed",
            "nunique",
        ),

        qwk_mean=(
            "qwk",
            "mean",
        ),

        qwk_std=(
            "qwk",
            "std",
        ),

        mae_mean=(
            "mae",
            "mean",
        ),

        mae_std=(
            "mae",
            "std",
        ),

        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),

        macro_f1_std=(
            "macro_f1",
            "std",
        ),

        accuracy_mean=(
            "accuracy",
            "mean",
        ),

        accuracy_std=(
            "accuracy",
            "std",
        ),
    )
)

aggregation_summary_by_candidate_path = (
    OUTPUT_ROOT
    / "aggregation_summary_by_candidate.csv"
)

atomic_csv_save(
    aggregation_summary_by_candidate_df,
    aggregation_summary_by_candidate_path,
    index=False,
)


# ============================================================
# 20. Top-two paired stratified patient bootstrap
# ============================================================

TOP_AGGREGATION_RULE = str(
    aggregation_ranking_df
    .iloc[0][
        "aggregation_rule"
    ]
)

SECOND_AGGREGATION_RULE = str(
    aggregation_ranking_df
    .iloc[1][
        "aggregation_rule"
    ]
)

source_ids = (
    source_file_df[
        "source_id"
    ]
    .tolist()
)

base_patient_df = (
    reference_audit_df[
        [
            "patient_id",
            PRIMARY_REFERENCE_RULE,
        ]
    ]
    .sort_values("patient_id")
    .reset_index(drop=True)
)

patient_ids = (
    base_patient_df[
        "patient_id"
    ]
    .to_numpy(
        dtype=int
    )
)

primary_y_true = (
    base_patient_df[
        PRIMARY_REFERENCE_RULE
    ]
    .to_numpy(
        dtype=int
    )
)

prediction_lookup = {}

for source_id in source_ids:
    source_patient_df = (
        patient_predictions_long_df[
            patient_predictions_long_df[
                "source_id"
            ]
            ==
            source_id
        ]
    )

    for aggregation_rule in [
        TOP_AGGREGATION_RULE,
        SECOND_AGGREGATION_RULE,
    ]:
        rule_df = (
            source_patient_df[
                source_patient_df[
                    "aggregation_rule"
                ]
                ==
                aggregation_rule
            ]
            .sort_values("patient_id")
            .reset_index(drop=True)
        )

        if not np.array_equal(
            rule_df[
                "patient_id"
            ].to_numpy(
                dtype=int
            ),
            patient_ids,
        ):
            raise RuntimeError(
                "Patient alignment failure for "
                f"{source_id}, {aggregation_rule}."
            )

        prediction_lookup[
            (
                source_id,
                aggregation_rule,
            )
        ] = (
            rule_df["y_pred"]
            .to_numpy(
                dtype=int
            )
        )


def mean_metrics_across_sources(
    sampled_indices,
    aggregation_rule: str,
):
    metric_values = {
        "qwk": [],
        "mae": [],
        "macro_f1": [],
        "accuracy": [],
    }

    sampled_true = (
        primary_y_true[
            sampled_indices
        ]
    )

    for source_id in source_ids:
        sampled_pred = (
            prediction_lookup[
                (
                    source_id,
                    aggregation_rule,
                )
            ][
                sampled_indices
            ]
        )

        metrics = compute_metrics(
            sampled_true,
            sampled_pred,
        )

        for metric_name in metric_values:
            metric_values[
                metric_name
            ].append(
                metrics[
                    metric_name
                ]
            )

    return {
        metric_name: float(
            np.mean(values)
        )

        for metric_name, values in (
            metric_values.items()
        )
    }


class_indices = {
    class_index: np.flatnonzero(
        primary_y_true
        ==
        class_index
    )

    for class_index in range(
        NUM_CLASSES
    )
}

for class_index, indices in (
    class_indices.items()
):
    if len(indices) == 0:
        raise RuntimeError(
            "Bootstrap class is empty: "
            f"{class_index}"
        )

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_rows = []

for bootstrap_index in range(
    BOOTSTRAP_REPETITIONS
):
    sampled_parts = []

    for class_index in range(
        NUM_CLASSES
    ):
        indices = class_indices[
            class_index
        ]

        sampled_parts.append(
            rng.choice(
                indices,
                size=len(indices),
                replace=True,
            )
        )

    sampled_indices = np.concatenate(
        sampled_parts
    )

    top_metrics = (
        mean_metrics_across_sources(
            sampled_indices=(
                sampled_indices
            ),

            aggregation_rule=(
                TOP_AGGREGATION_RULE
            ),
        )
    )

    second_metrics = (
        mean_metrics_across_sources(
            sampled_indices=(
                sampled_indices
            ),

            aggregation_rule=(
                SECOND_AGGREGATION_RULE
            ),
        )
    )

    bootstrap_rows.append({
        "bootstrap_index": int(
            bootstrap_index
        ),

        "top_qwk": (
            top_metrics["qwk"]
        ),

        "second_qwk": (
            second_metrics["qwk"]
        ),

        "delta_qwk": (
            top_metrics["qwk"]
            -
            second_metrics["qwk"]
        ),

        "top_mae": (
            top_metrics["mae"]
        ),

        "second_mae": (
            second_metrics["mae"]
        ),

        "delta_mae": (
            top_metrics["mae"]
            -
            second_metrics["mae"]
        ),

        "top_macro_f1": (
            top_metrics["macro_f1"]
        ),

        "second_macro_f1": (
            second_metrics["macro_f1"]
        ),

        "delta_macro_f1": (
            top_metrics["macro_f1"]
            -
            second_metrics["macro_f1"]
        ),

        "top_accuracy": (
            top_metrics["accuracy"]
        ),

        "second_accuracy": (
            second_metrics["accuracy"]
        ),

        "delta_accuracy": (
            top_metrics["accuracy"]
            -
            second_metrics["accuracy"]
        ),
    })

    if (
        bootstrap_index + 1
    ) % 500 == 0:
        print(
            "Completed paired bootstrap replicates:",
            bootstrap_index + 1,
            "/",
            BOOTSTRAP_REPETITIONS,
        )

top2_bootstrap_samples_df = pd.DataFrame(
    bootstrap_rows
)

top_point_row = (
    aggregation_ranking_df[
        aggregation_ranking_df[
            "aggregation_rule"
        ]
        ==
        TOP_AGGREGATION_RULE
    ]
    .iloc[0]
)

second_point_row = (
    aggregation_ranking_df[
        aggregation_ranking_df[
            "aggregation_rule"
        ]
        ==
        SECOND_AGGREGATION_RULE
    ]
    .iloc[0]
)

metric_columns = {
    "qwk": {
        "top_column": "top_qwk",
        "second_column": "second_qwk",
        "delta_column": "delta_qwk",
        "top_point": float(
            top_point_row[
                "qwk_mean"
            ]
        ),
        "second_point": float(
            second_point_row[
                "qwk_mean"
            ]
        ),
        "positive_favors_top": True,
    },

    "mae": {
        "top_column": "top_mae",
        "second_column": "second_mae",
        "delta_column": "delta_mae",
        "top_point": float(
            top_point_row[
                "mae_mean"
            ]
        ),
        "second_point": float(
            second_point_row[
                "mae_mean"
            ]
        ),
        "positive_favors_top": False,
    },

    "macro_f1": {
        "top_column": "top_macro_f1",
        "second_column": "second_macro_f1",
        "delta_column": "delta_macro_f1",
        "top_point": float(
            top_point_row[
                "macro_f1_mean"
            ]
        ),
        "second_point": float(
            second_point_row[
                "macro_f1_mean"
            ]
        ),
        "positive_favors_top": True,
    },

    "accuracy": {
        "top_column": "top_accuracy",
        "second_column": "second_accuracy",
        "delta_column": "delta_accuracy",
        "top_point": float(
            top_point_row[
                "accuracy_mean"
            ]
        ),
        "second_point": float(
            second_point_row[
                "accuracy_mean"
            ]
        ),
        "positive_favors_top": True,
    },
}

top2_bootstrap_summary_rows = []

for metric_name, specification in (
    metric_columns.items()
):
    top_values = (
        top2_bootstrap_samples_df[
            specification[
                "top_column"
            ]
        ]
        .dropna()
        .to_numpy()
    )

    second_values = (
        top2_bootstrap_samples_df[
            specification[
                "second_column"
            ]
        ]
        .dropna()
        .to_numpy()
    )

    delta_values = (
        top2_bootstrap_samples_df[
            specification[
                "delta_column"
            ]
        ]
        .dropna()
        .to_numpy()
    )

    probability_nonpositive = float(
        np.mean(
            delta_values <= 0
        )
    )

    probability_nonnegative = float(
        np.mean(
            delta_values >= 0
        )
    )

    two_sided_p = min(
        1.0,
        2.0
        *
        min(
            probability_nonpositive,
            probability_nonnegative,
        ),
    )

    top2_bootstrap_summary_rows.append({
        "metric": metric_name,

        "top_aggregation_rule": (
            TOP_AGGREGATION_RULE
        ),

        "second_aggregation_rule": (
            SECOND_AGGREGATION_RULE
        ),

        "top_point_estimate": (
            specification[
                "top_point"
            ]
        ),

        "top_ci_low": float(
            np.percentile(
                top_values,
                2.5,
            )
        ),

        "top_ci_high": float(
            np.percentile(
                top_values,
                97.5,
            )
        ),

        "second_point_estimate": (
            specification[
                "second_point"
            ]
        ),

        "second_ci_low": float(
            np.percentile(
                second_values,
                2.5,
            )
        ),

        "second_ci_high": float(
            np.percentile(
                second_values,
                97.5,
            )
        ),

        "difference_definition": (
            "Top aggregation rule minus "
            "second aggregation rule"
        ),

        "point_difference": float(
            specification[
                "top_point"
            ]
            -
            specification[
                "second_point"
            ]
        ),

        "difference_ci_low": float(
            np.percentile(
                delta_values,
                2.5,
            )
        ),

        "difference_ci_high": float(
            np.percentile(
                delta_values,
                97.5,
            )
        ),

        "bootstrap_two_sided_p": float(
            two_sided_p
        ),

        "direction_note": (
            "Positive favors the top rule"
            if specification[
                "positive_favors_top"
            ]
            else
            "Negative favors the top rule"
        ),
    })

top2_bootstrap_summary_df = pd.DataFrame(
    top2_bootstrap_summary_rows
)

top2_bootstrap_samples_path = (
    OUTPUT_ROOT
    / "aggregation_top2_paired_bootstrap_samples.csv"
)

top2_bootstrap_summary_path = (
    OUTPUT_ROOT
    / "aggregation_top2_paired_bootstrap_summary.csv"
)

atomic_csv_save(
    top2_bootstrap_samples_df,
    top2_bootstrap_samples_path,
    index=False,
)

atomic_csv_save(
    top2_bootstrap_summary_df,
    top2_bootstrap_summary_path,
    index=False,
)


# ============================================================
# 21. Lock the selected aggregation rule
# ============================================================

qwk_bootstrap_row = (
    top2_bootstrap_summary_df[
        top2_bootstrap_summary_df[
            "metric"
        ]
        ==
        "qwk"
    ]
    .iloc[0]
)

qwk_difference_ci_includes_zero = bool(
    qwk_bootstrap_row[
        "difference_ci_low"
    ]
    <=
    0
    <=
    qwk_bootstrap_row[
        "difference_ci_high"
    ]
)

locked_aggregation = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "protocol_version": (
        PROTOCOL_VERSION
    ),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "locked_aggregation_rule": (
        TOP_AGGREGATION_RULE
    ),

    "locked_aggregation_display_name": (
        AGGREGATION_RULES[
            TOP_AGGREGATION_RULE
        ]["display_name"]
    ),

    "locked_aggregation_description": (
        AGGREGATION_RULES[
            TOP_AGGREGATION_RULE
        ]["description"]
    ),

    "primary_reference_rule": (
        PRIMARY_REFERENCE_RULE
    ),

    "selection_hierarchy": (
        SELECTION_HIERARCHY
    ),

    "n_oof_sources": 6,

    "source_candidates": (
        CANDIDATES
    ),

    "source_seeds": (
        SEEDS
    ),

    "top_rule_mean_metrics": {
        "qwk": float(
            top_point_row[
                "qwk_mean"
            ]
        ),

        "mae": float(
            top_point_row[
                "mae_mean"
            ]
        ),

        "macro_f1": float(
            top_point_row[
                "macro_f1_mean"
            ]
        ),

        "accuracy": float(
            top_point_row[
                "accuracy_mean"
            ]
        ),
    },

    "second_rule": (
        SECOND_AGGREGATION_RULE
    ),

    "paired_qwk_difference": float(
        qwk_bootstrap_row[
            "point_difference"
        ]
    ),

    "paired_qwk_difference_95_ci": [
        float(
            qwk_bootstrap_row[
                "difference_ci_low"
            ]
        ),

        float(
            qwk_bootstrap_row[
                "difference_ci_high"
            ]
        ),
    ],

    "paired_qwk_difference_ci_includes_zero": (
        qwk_difference_ci_includes_zero
    ),

    "selection_interpretation": (
        "The aggregation rule was locked according to the "
        "prespecified development-only ranking hierarchy. "
        "A confidence interval including zero indicates that "
        "the top two rules were not clearly separated, but "
        "does not alter the prespecified procedural selection."
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

locked_aggregation_path = (
    OUTPUT_ROOT
    / "locked_patient_aggregation_rule.json"
)

atomic_json_save(
    locked_aggregation,
    locked_aggregation_path,
)


# ============================================================
# 22. Save locked-rule patient predictions
# ============================================================

locked_patient_predictions_df = (
    patient_predictions_long_df[
        patient_predictions_long_df[
            "aggregation_rule"
        ]
        ==
        TOP_AGGREGATION_RULE
    ]
    .copy()
    .sort_values(
        [
            "source_id",
            "patient_id",
        ]
    )
    .reset_index(drop=True)
)

locked_patient_predictions_path = (
    OUTPUT_ROOT
    / "locked_aggregation_patient_predictions.csv"
)

atomic_csv_save(
    locked_patient_predictions_df,
    locked_patient_predictions_path,
    index=False,
)


# ============================================================
# 23. Reference sensitivity for the locked aggregation
# ============================================================

reference_sensitivity_rows = []

for (
    source_id,
    candidate_name,
    training_seed,
), source_df in (
    locked_patient_predictions_df
    .groupby(
        [
            "source_id",
            "candidate_name",
            "training_seed",
        ]
    )
):
    for reference_rule in (
        REFERENCE_RULES
    ):
        metrics = compute_metrics(
            source_df[
                reference_rule
            ],
            source_df[
                "y_pred"
            ],
        )

        reference_sensitivity_rows.append({
            "source_id": source_id,
            "candidate_name": candidate_name,
            "training_seed": int(
                training_seed
            ),

            "aggregation_rule": (
                TOP_AGGREGATION_RULE
            ),

            "reference_rule": (
                reference_rule
            ),

            "reference_description": (
                REFERENCE_RULES[
                    reference_rule
                ]
            ),

            "analysis_role": (
                "primary"
                if reference_rule
                ==
                PRIMARY_REFERENCE_RULE
                else
                "sensitivity_only"
            ),

            "qwk": float(
                metrics["qwk"]
            ),

            "mae": float(
                metrics["mae"]
            ),

            "macro_f1": float(
                metrics["macro_f1"]
            ),

            "accuracy": float(
                metrics["accuracy"]
            ),
        })

reference_sensitivity_results_df = pd.DataFrame(
    reference_sensitivity_rows
)

reference_sensitivity_summary_df = (
    reference_sensitivity_results_df
    .groupby(
        [
            "aggregation_rule",
            "reference_rule",
            "reference_description",
            "analysis_role",
        ],
        as_index=False,
    )
    .agg(
        n_sources=(
            "source_id",
            "nunique",
        ),

        qwk_mean=(
            "qwk",
            "mean",
        ),

        qwk_std=(
            "qwk",
            "std",
        ),

        mae_mean=(
            "mae",
            "mean",
        ),

        mae_std=(
            "mae",
            "std",
        ),

        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),

        macro_f1_std=(
            "macro_f1",
            "std",
        ),

        accuracy_mean=(
            "accuracy",
            "mean",
        ),

        accuracy_std=(
            "accuracy",
            "std",
        ),
    )
)

reference_sensitivity_results_path = (
    OUTPUT_ROOT
    / "reference_sensitivity_results_by_source.csv"
)

reference_sensitivity_summary_path = (
    OUTPUT_ROOT
    / "reference_sensitivity_summary_locked_aggregation.csv"
)

atomic_csv_save(
    reference_sensitivity_results_df,
    reference_sensitivity_results_path,
    index=False,
)

atomic_csv_save(
    reference_sensitivity_summary_df,
    reference_sensitivity_summary_path,
    index=False,
)


# ============================================================
# 24. Full aggregation × reference sensitivity grid
# ============================================================

full_grid_rows = []

for (
    source_id,
    candidate_name,
    training_seed,
    aggregation_rule,
), group_df in (
    patient_predictions_long_df
    .groupby(
        [
            "source_id",
            "candidate_name",
            "training_seed",
            "aggregation_rule",
        ]
    )
):
    for reference_rule in (
        REFERENCE_RULES
    ):
        metrics = compute_metrics(
            group_df[
                reference_rule
            ],
            group_df[
                "y_pred"
            ],
        )

        full_grid_rows.append({
            "source_id": source_id,
            "candidate_name": candidate_name,
            "training_seed": int(
                training_seed
            ),

            "aggregation_rule": (
                aggregation_rule
            ),

            "aggregation_display_name": (
                AGGREGATION_RULES[
                    aggregation_rule
                ]["display_name"]
            ),

            "reference_rule": (
                reference_rule
            ),

            "analysis_role": (
                "primary_selection"
                if (
                    reference_rule
                    ==
                    PRIMARY_REFERENCE_RULE
                )
                else
                "descriptive_sensitivity_only"
            ),

            "qwk": float(
                metrics["qwk"]
            ),

            "mae": float(
                metrics["mae"]
            ),

            "macro_f1": float(
                metrics["macro_f1"]
            ),

            "accuracy": float(
                metrics["accuracy"]
            ),
        })

full_grid_results_df = pd.DataFrame(
    full_grid_rows
)

full_grid_summary_df = (
    full_grid_results_df
    .groupby(
        [
            "aggregation_rule",
            "aggregation_display_name",
            "reference_rule",
            "analysis_role",
        ],
        as_index=False,
    )
    .agg(
        n_sources=(
            "source_id",
            "nunique",
        ),

        qwk_mean=(
            "qwk",
            "mean",
        ),

        qwk_std=(
            "qwk",
            "std",
        ),

        mae_mean=(
            "mae",
            "mean",
        ),

        mae_std=(
            "mae",
            "std",
        ),

        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),

        macro_f1_std=(
            "macro_f1",
            "std",
        ),

        accuracy_mean=(
            "accuracy",
            "mean",
        ),

        accuracy_std=(
            "accuracy",
            "std",
        ),
    )
)

full_grid_results_path = (
    OUTPUT_ROOT
    / "aggregation_reference_grid_results_by_source.csv"
)

full_grid_summary_path = (
    OUTPUT_ROOT
    / "aggregation_reference_grid_summary.csv"
)

atomic_csv_save(
    full_grid_results_df,
    full_grid_results_path,
    index=False,
)

atomic_csv_save(
    full_grid_summary_df,
    full_grid_summary_path,
    index=False,
)


# ============================================================
# 25. Prediction disagreement among aggregation rules
# ============================================================

aggregation_disagreement_rows = []

aggregation_rule_names = list(
    AGGREGATION_RULES.keys()
)

for source_id in source_ids:
    source_df = (
        patient_predictions_long_df[
            patient_predictions_long_df[
                "source_id"
            ]
            ==
            source_id
        ]
    )

    source_candidate = (
        source_df[
            "candidate_name"
        ]
        .iloc[0]
    )

    source_seed = int(
        source_df[
            "training_seed"
        ]
        .iloc[0]
    )

    prediction_arrays = {}

    for aggregation_rule in (
        aggregation_rule_names
    ):
        rule_df = (
            source_df[
                source_df[
                    "aggregation_rule"
                ]
                ==
                aggregation_rule
            ]
            .sort_values("patient_id")
        )

        prediction_arrays[
            aggregation_rule
        ] = (
            rule_df["y_pred"]
            .to_numpy(
                dtype=int
            )
        )

    for rule_a, rule_b in combinations(
        aggregation_rule_names,
        2,
    ):
        disagreement = (
            prediction_arrays[
                rule_a
            ]
            !=
            prediction_arrays[
                rule_b
            ]
        )

        aggregation_disagreement_rows.append({
            "source_id": source_id,
            "candidate_name": (
                source_candidate
            ),
            "training_seed": (
                source_seed
            ),

            "aggregation_rule_a": (
                rule_a
            ),

            "aggregation_rule_b": (
                rule_b
            ),

            "n_disagree": int(
                disagreement.sum()
            ),

            "percent_disagree": float(
                100.0
                *
                disagreement.mean()
            ),
        })

aggregation_disagreement_df = pd.DataFrame(
    aggregation_disagreement_rows
)

aggregation_disagreement_summary_df = (
    aggregation_disagreement_df
    .groupby(
        [
            "aggregation_rule_a",
            "aggregation_rule_b",
        ],
        as_index=False,
    )
    .agg(
        mean_percent_disagree=(
            "percent_disagree",
            "mean",
        ),

        std_percent_disagree=(
            "percent_disagree",
            "std",
        ),

        min_percent_disagree=(
            "percent_disagree",
            "min",
        ),

        max_percent_disagree=(
            "percent_disagree",
            "max",
        ),
    )
)

aggregation_disagreement_path = (
    OUTPUT_ROOT
    / "aggregation_prediction_disagreement_by_source.csv"
)

aggregation_disagreement_summary_path = (
    OUTPUT_ROOT
    / "aggregation_prediction_disagreement_summary.csv"
)

atomic_csv_save(
    aggregation_disagreement_df,
    aggregation_disagreement_path,
    index=False,
)

atomic_csv_save(
    aggregation_disagreement_summary_df,
    aggregation_disagreement_summary_path,
    index=False,
)


# ============================================================
# 26. Patient-reference disagreement and distributions
# ============================================================

reference_disagreement_rows = []

for alternative_reference in [
    "modal_high",
    "median_low",
    "median_high",
    "maximum_label",
]:
    disagreement = (
        reference_audit_df[
            PRIMARY_REFERENCE_RULE
        ]
        !=
        reference_audit_df[
            alternative_reference
        ]
    )

    reference_disagreement_rows.append({
        "primary_reference": (
            PRIMARY_REFERENCE_RULE
        ),

        "alternative_reference": (
            alternative_reference
        ),

        "n_disagree": int(
            disagreement.sum()
        ),

        "percent_disagree": float(
            100.0
            *
            disagreement.mean()
        ),
    })

reference_disagreement_df = pd.DataFrame(
    reference_disagreement_rows
)

reference_distribution_rows = []

for reference_rule in (
    REFERENCE_RULES
):
    counts = (
        reference_audit_df[
            reference_rule
        ]
        .value_counts()
        .reindex(
            range(
                NUM_CLASSES
            ),
            fill_value=0,
        )
        .sort_index()
    )

    for class_index in range(
        NUM_CLASSES
    ):
        reference_distribution_rows.append({
            "reference_rule": (
                reference_rule
            ),

            "mayo_class": int(
                class_index
            ),

            "n_patients": int(
                counts.loc[
                    class_index
                ]
            ),

            "percent_patients": float(
                100.0
                *
                counts.loc[
                    class_index
                ]
                /
                len(
                    reference_audit_df
                )
            ),
        })

reference_distribution_df = pd.DataFrame(
    reference_distribution_rows
)

reference_disagreement_path = (
    OUTPUT_ROOT
    / "patient_reference_disagreement_summary.csv"
)

reference_distribution_path = (
    OUTPUT_ROOT
    / "patient_reference_distributions.csv"
)

atomic_csv_save(
    reference_disagreement_df,
    reference_disagreement_path,
    index=False,
)

atomic_csv_save(
    reference_distribution_df,
    reference_distribution_path,
    index=False,
)


# ============================================================
# 27. Focused maximum-severity compatibility table
# ============================================================

maximum_severity_compatibility_df = (
    full_grid_summary_df[
        full_grid_summary_df[
            "aggregation_rule"
        ]
        ==
        "maximum_predicted_class"
    ]
    .copy()
    .sort_values(
        "reference_rule"
    )
    .reset_index(drop=True)
)

maximum_severity_compatibility_path = (
    OUTPUT_ROOT
    / "maximum_prediction_reference_compatibility.csv"
)

atomic_csv_save(
    maximum_severity_compatibility_df,
    maximum_severity_compatibility_path,
    index=False,
)


# ============================================================
# 28. Save report-ready tables
# ============================================================

report_aggregation_table_df = (
    aggregation_ranking_df[
        [
            "development_rank",
            "aggregation_rule",
            "aggregation_display_name",
            "n_sources",

            "qwk_mean",
            "qwk_std",

            "mae_mean",
            "mae_std",

            "macro_f1_mean",
            "macro_f1_std",

            "accuracy_mean",
            "accuracy_std",

            "n_source_wins",
            "median_within_source_rank",
        ]
    ]
    .copy()
)

report_reference_table_df = (
    reference_sensitivity_summary_df[
        [
            "aggregation_rule",
            "reference_rule",
            "analysis_role",
            "n_sources",

            "qwk_mean",
            "qwk_std",

            "mae_mean",
            "mae_std",

            "macro_f1_mean",
            "macro_f1_std",

            "accuracy_mean",
            "accuracy_std",
        ]
    ]
    .copy()
)

report_aggregation_table_path = (
    OUTPUT_ROOT
    / "table_aggregation_primary_modal_low.csv"
)

report_reference_table_path = (
    OUTPUT_ROOT
    / "table_reference_sensitivity_locked_aggregation.csv"
)

atomic_csv_save(
    report_aggregation_table_df,
    report_aggregation_table_path,
    index=False,
)

atomic_csv_save(
    report_reference_table_df,
    report_reference_table_path,
    index=False,
)


# ============================================================
# 29. Compact decision summary
# ============================================================

decision_summary = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "protocol_version": (
        PROTOCOL_VERSION
    ),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,

    "primary_reference_rule": (
        PRIMARY_REFERENCE_RULE
    ),

    "locked_aggregation_rule": (
        TOP_AGGREGATION_RULE
    ),

    "locked_aggregation_display_name": (
        AGGREGATION_RULES[
            TOP_AGGREGATION_RULE
        ]["display_name"]
    ),

    "second_aggregation_rule": (
        SECOND_AGGREGATION_RULE
    ),

    "paired_top2_qwk_difference": float(
        qwk_bootstrap_row[
            "point_difference"
        ]
    ),

    "paired_top2_qwk_difference_95_ci": [
        float(
            qwk_bootstrap_row[
                "difference_ci_low"
            ]
        ),

        float(
            qwk_bootstrap_row[
                "difference_ci_high"
            ]
        ),
    ],

    "paired_top2_qwk_ci_includes_zero": (
        qwk_difference_ci_includes_zero
    ),

    "selection_note": (
        "Aggregation was selected solely against modal_low "
        "using the prespecified development-only hierarchy. "
        "Alternative patient references were evaluated only "
        "as sensitivity analyses."
    ),

    "maximum_severity_note": (
        "Maximum predicted image severity and maximum-label "
        "patient reference represent a worst-segment-oriented "
        "definition, whereas modal_low represents the most "
        "frequent image-level label. Performance differences "
        "between these combinations should therefore be "
        "interpreted as target-definition sensitivity rather "
        "than direct evidence of clinical invalidity."
    ),

    "output_files": {
        "aggregation_ranking": str(
            aggregation_ranking_path
        ),

        "paired_top2_bootstrap": str(
            top2_bootstrap_summary_path
        ),

        "locked_aggregation": str(
            locked_aggregation_path
        ),

        "reference_sensitivity": str(
            reference_sensitivity_summary_path
        ),

        "full_grid": str(
            full_grid_summary_path
        ),

        "locked_predictions": str(
            locked_patient_predictions_path
        ),
    },
}

decision_summary_path = (
    OUTPUT_ROOT
    / "aggregation_reference_decision_summary.json"
)

atomic_json_save(
    decision_summary,
    decision_summary_path,
)


# ============================================================
# 30. Display results
# ============================================================

print("\n" + "=" * 120)
print("PRIMARY AGGREGATION RANKING AGAINST MODAL_LOW")
print("=" * 120)

display(
    report_aggregation_table_df
)


print("\n" + "=" * 120)
print("TOP-TWO AGGREGATION RULES: PAIRED PATIENT BOOTSTRAP")
print("=" * 120)

display(
    top2_bootstrap_summary_df
)


print("\n" + "=" * 120)
print("LOCKED PATIENT AGGREGATION RULE")
print("=" * 120)

print(
    "Locked rule:",
    TOP_AGGREGATION_RULE,
)

print(
    "Display name:",
    AGGREGATION_RULES[
        TOP_AGGREGATION_RULE
    ]["display_name"],
)

print(
    "Second-ranked rule:",
    SECOND_AGGREGATION_RULE,
)

print(
    "Paired QWK difference CI includes zero:",
    qwk_difference_ci_includes_zero,
)


print("\n" + "=" * 120)
print("REFERENCE SENSITIVITY FOR THE LOCKED AGGREGATION")
print("=" * 120)

display(
    report_reference_table_df
)


print("\n" + "=" * 120)
print("MAXIMUM-PREDICTION REFERENCE COMPATIBILITY")
print("=" * 120)

display(
    maximum_severity_compatibility_df[
        [
            "aggregation_rule",
            "reference_rule",
            "analysis_role",
            "qwk_mean",
            "qwk_std",
            "mae_mean",
            "mae_std",
            "macro_f1_mean",
            "accuracy_mean",
        ]
    ]
)


print("\n" + "=" * 120)
print("PATIENT-REFERENCE DISAGREEMENT")
print("=" * 120)

display(
    reference_disagreement_df
)


# ============================================================
# 31. Final status
# ============================================================

print("\n" + "=" * 120)
print("03_patient_aggregation_and_reference_sensitivity COMPLETED")
print("=" * 120)

print(
    "Output directory:",
    OUTPUT_ROOT,
)

print(
    "Frozen analysis plan:",
    analysis_plan_path,
)

print(
    "Aggregation ranking:",
    aggregation_ranking_path,
)

print(
    "Paired top-two bootstrap:",
    top2_bootstrap_summary_path,
)

print(
    "Locked aggregation rule:",
    locked_aggregation_path,
)

print(
    "Reference sensitivity summary:",
    reference_sensitivity_summary_path,
)

print(
    "Full aggregation-reference grid:",
    full_grid_summary_path,
)

print(
    "Locked aggregation predictions:",
    locked_patient_predictions_path,
)

print(
    "Decision summary:",
    decision_summary_path,
)

print(
    "\nOfficial LIMUC test data loaded:",
    False,
)

print(
    "Official LIMUC test data evaluated:",
    False,
)

print(
    "\nNo model training was performed in this notebook."
)